In [25]:
!pip install xgboost lightgbm catboost joblib --quiet

In [26]:
import pandas as pd
import numpy as np
import joblib
import xgboost as xgb
import warnings
from datetime import datetime
warnings.filterwarnings("ignore")

In [27]:
df_main = pd.read_csv("content/weekly_features_engineered_v3.csv")
df_main["Date"] = pd.to_datetime(df_main["Date"])
df_main = df_main.sort_values("Date")
df_main.replace([np.inf, -np.inf], np.nan, inplace=True)
df_main = df_main.dropna().reset_index(drop=True)

print("Dataset shape:", df_main.shape)
print("Date range:", df_main["Date"].min(), "→", df_main["Date"].max())
print("Vegetables:", df_main["Vegetable"].unique().tolist())

Dataset shape: (4801, 63)
Date range: 2010-01-11 00:00:00 → 2026-02-23 00:00:00
Vegetables: ['Brinjals', 'Cabbage', 'Carrot', 'Pumpkin', 'Tomatoes', 'Bitter Gourd']


In [28]:
# Load best tuned model saved from tuning notebook
cat_model_tuned = joblib.load("content/cat_tuned-new.pkl")

best_model      = cat_model_tuned
best_model_name = "CatBoost"

print("Model loaded:", best_model_name)

Model loaded: CatBoost


In [29]:
# Arosha's wholesale predictions — one row per vegetable
arosha_pred = pd.read_csv("content/Predictions.csv")
arosha_pred['week_num'] = arosha_pred['Week'].str.replace('W', '').astype(int)

# Amika's weather predictions — daily
amika_pred = pd.read_csv("content/amika.csv")
amika_pred['date'] = pd.to_datetime(amika_pred['date'])

# Fix: Calculate week number based on the date
def get_week_number(date):
    # Week 1 starts on Jan 1
    year_start = pd.Timestamp(year=date.year, month=1, day=1)

    # Calculate days since start of year
    days_since_start = (date - year_start).days

    # Week number (1-indexed)
    week_num = (days_since_start // 7) + 1

    # Handle the last week of year (52 or 53)
    if week_num > 52:
        week_num = 52

    return week_num

# Apply the function to calculate week_num for each date
amika_pred['week_num'] = amika_pred['date'].apply(get_week_number)

# Aggregate all 7 cities and 7 days to a single weekly average
amika_weekly = amika_pred.groupby('week_num').agg(
    avg_flood_prob   = ('prob_flood_risk', 'mean'),
    avg_drought_prob = ('prob_drought',    'mean')
).reset_index()

print("Arosha wholesale predictions loaded:", arosha_pred.shape)
print("Amika weather predictions loaded:", amika_pred.shape)
print("Aggregated weather for week_num:", amika_weekly['week_num'].tolist())
print(amika_weekly)

Arosha wholesale predictions loaded: (12, 5)
Amika weather predictions loaded: (49, 7)
Aggregated weather for week_num: [11, 12]
   week_num  avg_flood_prob  avg_drought_prob
0        11        0.002905          0.220857
1        12        0.011464          0.173000


In [30]:
def get_week_from_date(date_obj):
    week_ranges = [
        (1,1,1,7), (1,8,1,14), (1,15,1,21), (1,22,1,28), (1,29,2,4),
        (2,5,2,11), (2,12,2,18), (2,19,2,25), (2,26,3,4), (3,5,3,11),
        (3,12,3,18), (3,19,3,25), (3,26,4,1), (4,2,4,8), (4,9,4,15),
        (4,16,4,22), (4,23,4,29), (4,30,5,6), (5,7,5,13), (5,14,5,20),
        (5,21,5,27), (5,28,6,3), (6,4,6,10), (6,11,6,17), (6,18,6,24),
        (6,25,7,1), (7,2,7,8), (7,9,7,15), (7,16,7,22), (7,23,7,29),
        (7,30,8,5), (8,6,8,12), (8,13,8,19), (8,20,8,26), (8,27,9,2),
        (9,3,9,9), (9,10,9,16), (9,17,9,23), (9,24,9,30), (10,1,10,7),
        (10,8,10,14), (10,15,10,21), (10,22,10,28), (10,29,11,4),
        (11,5,11,11), (11,12,11,18), (11,19,11,25), (11,26,12,2),
        (12,3,12,9), (12,10,12,16), (12,17,12,23), (12,24,12,31)
    ]
    week_start_dates = [
        "2026-01-01","2026-01-08","2026-01-15","2026-01-22","2026-01-29",
        "2026-02-05","2026-02-12","2026-02-19","2026-02-26","2026-03-05",
        "2026-03-12","2026-03-19","2026-03-26","2026-04-02","2026-04-09",
        "2026-04-16","2026-04-23","2026-04-30","2026-05-07","2026-05-14",
        "2026-05-21","2026-05-28","2026-06-04","2026-06-11","2026-06-18",
        "2026-06-25","2026-07-02","2026-07-09","2026-07-16","2026-07-23",
        "2026-07-30","2026-08-06","2026-08-13","2026-08-20","2026-08-27",
        "2026-09-03","2026-09-10","2026-09-17","2026-09-24","2026-10-01",
        "2026-10-08","2026-10-15","2026-10-22","2026-10-29","2026-11-05",
        "2026-11-12","2026-11-19","2026-11-26","2026-12-03","2026-12-10",
        "2026-12-17","2026-12-24"
    ]

    today = date_obj
    for week_num, (sm, sd, em, ed) in enumerate(week_ranges, 1):
        start = datetime(today.year, sm, sd)
        end   = datetime(today.year, em, ed)
        if start <= today <= end:
            return week_num, pd.Timestamp(week_start_dates[week_num - 1])
    return None, None

today                    = datetime.now()
current_week, week_start = get_week_from_date(today)

future_dates = pd.to_datetime([week_start])
vegetables   = df_main["Vegetable"].unique()

print(f"Today's date : {today.strftime('%Y-%m-%d')}")
print(f"Current week : W{current_week}")
print(f"Week start   : {week_start.strftime('%Y-%m-%d')}")
print(f"Predicting for vegetables: {vegetables.tolist()}")

Today's date : 2026-03-16
Current week : W11
Week start   : 2026-03-12
Predicting for vegetables: ['Brinjals', 'Cabbage', 'Carrot', 'Pumpkin', 'Tomatoes', 'Bitter Gourd']


In [31]:
future_rows = []

for veg in vegetables:
    veg_df = df_main[df_main["Vegetable"] == veg].sort_values("Date").copy()

    for date in future_dates:
        last_row   = veg_df.iloc[-1].copy()
        last_price = last_row["Price"] if len(future_rows) == 0 else future_rows[-1]["Predicted Price"]

        new_row         = last_row.copy()
        new_row["Date"] = date
        week_num        = current_week
        year            = date.year

        # Pipeline Input 1 — Arosha's wholesale prediction
        arosha_row = arosha_pred[
            (arosha_pred['Year']      == year) &
            (arosha_pred['week_num']  == week_num) &
            (arosha_pred['Vegetable'] == veg)
        ]
        if not arosha_row.empty:
            wp = arosha_row['Predicted Price'].values[0]
            new_row["Wholesale_Price"]         = wp
            new_row["Wholesale_Lag1"]          = last_row["Wholesale_Price"]
            new_row["Wholesale_Lag2"]          = last_row["Wholesale_Lag1"]
            new_row["Wholesale_Rolling_Mean4"] = (
                wp + last_row["Wholesale_Price"] +
                last_row["Wholesale_Lag1"] + last_row["Wholesale_Lag2"]
            ) / 4
        else:
            print(f"WARNING: No wholesale data found for {veg} Week {week_num} {year}")

        # Pipeline Input 2 — Amika's weather prediction
        amika_row = amika_weekly[amika_weekly['week_num'] == week_num]
        if not amika_row.empty:
            new_row["avg_flood_prob"]   = amika_row['avg_flood_prob'].values[0]
            new_row["avg_drought_prob"] = amika_row['avg_drought_prob'].values[0]

        # Roll price lag features forward (largest first so we don't overwrite before reading)
        if "Price_Lag_4" in new_row.index: new_row["Price_Lag_4"] = new_row["Price_Lag_3"]
        if "Price_Lag_3" in new_row.index: new_row["Price_Lag_3"] = new_row["Price_Lag_2"]
        if "Price_Lag_2" in new_row.index: new_row["Price_Lag_2"] = new_row["Price_Lag_1"]
        if "Price_Lag_1" in new_row.index: new_row["Price_Lag_1"] = last_price

        # Update rolling means using the newly rolled lags
        if "Rolling_Mean_4" in new_row.index:
            new_row["Rolling_Mean_4"] = (
                new_row["Price_Lag_1"] + new_row["Price_Lag_2"] +
                new_row["Price_Lag_3"] + new_row["Price_Lag_4"]
            ) / 4
        if "Rolling_Mean_8" in new_row.index:
            new_row["Rolling_Mean_8"] = (
                new_row["Rolling_Mean_4"] + last_row["Rolling_Mean_4"]
            ) / 2

        # Update time features
        new_row["Month"]        = date.month
        new_row["Week_of_Year"] = week_num
        new_row["Year"]         = year
        new_row["Quarter"]      = (date.month - 1) // 3 + 1

        # Predict
        X_new = pd.DataFrame([new_row.drop(["Date", "Vegetable", "Price"])])
        if best_model_name == "XGBoost":
            pred_price = best_model.predict(xgb.DMatrix(X_new))[0]
        else:
            pred_price = best_model.predict(X_new)[0]

        future_rows.append({
            "Vegetable":       veg,
            "Week":            f"W{current_week}",
            "Year":            year,
            "Predicted Price": round(pred_price, 2),
            "Wholesale_Input": round(new_row.get("Wholesale_Price", 0), 2),
            "Flood_Risk":      round(new_row.get("avg_flood_prob", 0), 4),
            "Drought_Risk":    round(new_row.get("avg_drought_prob", 0), 4)
        })

        new_row["Price"] = pred_price
        veg_df = pd.concat([veg_df, pd.DataFrame([new_row])], ignore_index=True)

future_output = pd.DataFrame(future_rows)
future_output = future_output.sort_values("Vegetable").reset_index(drop=True)
print(future_output.to_string())

      Vegetable Week  Year  Predicted Price  Wholesale_Input  Flood_Risk  Drought_Risk
0  Bitter Gourd  W11  2026           444.96           442.48      0.0029        0.2209
1      Brinjals  W11  2026           336.43           170.17      0.0029        0.2209
2       Cabbage  W11  2026           337.07           148.84      0.0029        0.2209
3        Carrot  W11  2026           379.84           211.05      0.0029        0.2209
4       Pumpkin  W11  2026           256.78           105.15      0.0029        0.2209
5      Tomatoes  W11  2026           257.49           145.46      0.0029        0.2209


In [32]:
future_output.to_csv("content/future_price_predictions-market.csv", index=False)

print("Saved: /content/future_price_predictions-market.csv")
print("Total predictions:", len(future_output))
print(f"Period: Week {current_week} of 2026")
print("Vegetables:", future_output["Vegetable"].nunique())
print()
print("Pipeline inputs used:")
print("  Wholesale prices → Arosha's model output")
print("  Weather risk     → Amika's model output")

Saved: /content/future_price_predictions-market.csv
Total predictions: 6
Period: Week 11 of 2026
Vegetables: 6

Pipeline inputs used:
  Wholesale prices → Arosha's model output
  Weather risk     → Amika's model output
